# Stanford Cars: Hybrid Quantum Neural Network (HQNN)

Trains a **PennyLane hybrid head** on a frozen ResNet backbone loaded from **`stanford_cars_resnet.pth`**.

That checkpoint uses a **custom ResNet** with **1024-D** pooled features and 196 classes.


In [1]:
from pathlib import Path
from typing import Optional

import numpy as np
import pennylane as qml
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from torchvision.datasets import StanfordCars
from tqdm import tqdm

torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


## Dataset: Stanford Cars

In [2]:
# Same mean/std as stanford_cars/restnet_stanford_cars.ipynb
stats = ((0.4708, 0.4602, 0.4550), (0.2891, 0.2882, 0.2967))

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(*stats),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(*stats),
])


import platform
import shutil
import subprocess

# Manual install layout:
# ../archive/car_devkit/devkit
# ../archive/cars_train
# ../archive/cars_test
ARCHIVE_DIR = Path("../archive")


def _ensure_link(link: Path, target: Path) -> None:
    target = target.resolve()
    if not target.exists():
        return
    if link.exists():
        return
    link.parent.mkdir(parents=True, exist_ok=True)
    try:
        link.symlink_to(target, target_is_directory=target.is_dir())
    except OSError:
        if platform.system() == "Windows" and target.is_dir():
            subprocess.check_call(
                ["cmd", "/c", "mklink", "/J", str(link.resolve()), str(target)],
                cwd=str(link.parent),
            )
        else:
            raise


def _ensure_file(target_file: Path, source_file: Path, overwrite: bool = False) -> None:
    source_file = source_file.resolve()
    if not source_file.is_file():
        return
    if target_file.exists() and not overwrite:
        return
    if target_file.exists() and overwrite:
        target_file.unlink()
    target_file.parent.mkdir(parents=True, exist_ok=True)
    try:
        target_file.symlink_to(source_file)
    except OSError:
        shutil.copy2(source_file, target_file)


def _has_class_labels(mat_path: Path) -> bool:
    if not mat_path.is_file():
        return False
    try:
        import scipy.io as sio

        ann = sio.loadmat(mat_path, squeeze_me=True).get("annotations")
        if ann is None or getattr(ann, "dtype", None) is None:
            return False
        names = ann.dtype.names or ()
        return "class" in names
    except Exception:
        return False


def resolve_stanford_cars_root() -> Path:
    devkit_root = ARCHIVE_DIR / "car_devkit"
    devkit_dir = devkit_root / "devkit"
    train_dir = ARCHIVE_DIR / "cars_train"
    test_dir = ARCHIVE_DIR / "cars_test"

    if not (devkit_dir / "cars_meta.mat").is_file():
        raise RuntimeError(
            f"Could not find devkit at {devkit_dir}. Expected cars_meta.mat there."
        )
    if not train_dir.is_dir() or not test_dir.is_dir():
        raise RuntimeError(
            f"Could not find image dirs at {ARCHIVE_DIR}. Expected cars_train/ and cars_test/."
        )

    canonical = ARCHIVE_DIR / "stanford_cars"
    if canonical.is_symlink():
        canonical.unlink()
    canonical.mkdir(parents=True, exist_ok=True)
    _ensure_link(canonical / "devkit", devkit_dir)
    _ensure_link(canonical / "cars_train", train_dir)
    _ensure_link(canonical / "cars_test", test_dir)

    canonical_test_ann = canonical / "cars_test_annos_withlabels.mat"
    source_test_ann = devkit_root / "cars_test_annos_withlabels.mat"
    _ensure_file(canonical_test_ann, source_test_ann)
    if not _has_class_labels(canonical_test_ann):
        _ensure_file(canonical_test_ann, source_test_ann, overwrite=True)

    if not (canonical / "devkit" / "cars_meta.mat").is_file():
        raise RuntimeError(
            f"Expected {canonical}/devkit/cars_meta.mat."
        )
    if not (canonical / "cars_train").is_dir() or not (canonical / "cars_test").is_dir():
        raise RuntimeError(
            f"Expected {canonical}/cars_train and {canonical}/cars_test."
        )
    if not _has_class_labels(canonical / "cars_test_annos_withlabels.mat"):
        raise RuntimeError(
            f"Expected labeled test annotations at {canonical}/cars_test_annos_withlabels.mat."
        )

    return ARCHIVE_DIR.resolve()


data_root = resolve_stanford_cars_root()
print("StanfordCars root:", data_root)
batch_size = 64

train_ds = StanfordCars(
    root=data_root, split="train", download=False, transform=train_transform
)
test_ds = StanfordCars(
    root=data_root, split="test", download=False, transform=test_transform
)

num_classes = len(train_ds.classes)
print("num_classes:", num_classes)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)
test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)


RuntimeError: Expected labeled test annotations at ..\archive\stanford_cars/cars_test_annos_withlabels.mat.

## Stanford ResNet `restnet_stanford_cars.ipynb`

1024-D embedding before fc; forward_features returns that vector.


In [ ]:
def conv_shortcut(in_channel, out_channel, stride):
    layers = [
        nn.Conv2d(in_channel, out_channel, kernel_size=(1, 1), stride=(stride, stride)),
        nn.BatchNorm2d(out_channel),
    ]
    return nn.Sequential(*layers)


def block(in_channel, out_channel, k_size, stride, conv=False):
    first_layers = [
        nn.Conv2d(in_channel, out_channel[0], kernel_size=(1, 1), stride=(1, 1)),
        nn.BatchNorm2d(out_channel[0]),
        nn.ReLU(inplace=True),
    ]
    if conv:
        first_layers[0].stride = (stride, stride)

    second_layers = [
        nn.Conv2d(
            out_channel[0],
            out_channel[1],
            kernel_size=(k_size, k_size),
            stride=(1, 1),
            padding=1,
        ),
        nn.BatchNorm2d(out_channel[1]),
    ]

    return nn.Sequential(*first_layers + second_layers)


class StanfordResNet(nn.Module):

    def __init__(self, in_channels: int, num_classes: int):
        super().__init__()
        self.stg1 = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=64, kernel_size=(3), stride=(1), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.convShortcut2 = conv_shortcut(64, 256, 1)
        self.conv2 = block(64, [64, 256], 3, 1, conv=True)
        self.ident2 = block(256, [64, 256], 3, 1)

        self.convShortcut3 = conv_shortcut(256, 512, 2)
        self.conv3 = block(256, [128, 512], 3, 2, conv=True)
        self.ident3 = block(512, [128, 512], 3, 2)

        self.convShortcut4 = conv_shortcut(512, 1024, 2)
        self.conv4 = block(512, [256, 1024], 3, 2, conv=True)
        self.ident4 = block(1024, [256, 1024], 3, 2)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(1024, num_classes)

    def forward_features(self, inputs: torch.Tensor) -> torch.Tensor:
        out = self.stg1(inputs)
        out = F.relu(self.conv2(out) + self.convShortcut2(out))
        out = F.relu(self.ident2(out) + out)
        out = F.relu(self.ident2(out) + out)
        out = F.relu(self.ident2(out) + out)
        out = F.relu(self.conv3(out) + self.convShortcut3(out))
        out = F.relu(self.ident3(out) + out)
        out = F.relu(self.ident3(out) + out)
        out = F.relu(self.ident3(out) + out)
        out = F.relu(self.ident3(out) + out)
        out = F.relu(self.conv4(out) + self.convShortcut4(out))
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = F.relu(self.ident4(out) + out)
        out = self.pool(out)
        return torch.flatten(out, 1)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.fc(self.forward_features(inputs))


## Hybrid head + load `stanford_cars_resnet.pth`

Replace fc with: Linear(1024 → adapter) → ReLU → Linear(adapter → n_qubits) → VQC → Linear(n_qubits → num_classes).

A baseline_head Linear(1024, num_classes) copies the pretrained fc weights for evaluation (frozen backbone + classical head).


In [ ]:
def make_quantum_layer(n_qubits: int, n_layers: int):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="X")
        qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    weight_shapes = {"weights": (n_layers, n_qubits, 3)}
    return qml.qnn.TorchLayer(circuit, weight_shapes)


class HybridHead(nn.Module):
    def __init__(
        self,
        in_dim: int,
        num_classes: int,
        n_qubits: int,
        n_layers: int,
        adapter_dim: int = 128,
    ):
        super().__init__()
        self.adapter = nn.Sequential(
            nn.Linear(in_dim, adapter_dim),
            nn.ReLU(inplace=True),
            nn.Linear(adapter_dim, n_qubits),
        )
        self.scale = nn.Parameter(torch.ones(n_qubits))
        self.qlayer = make_quantum_layer(n_qubits, n_layers)
        self.head = nn.Linear(n_qubits, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.adapter(x)
        angles = torch.tanh(h) * self.scale * np.pi
        q_out = self.qlayer(angles)
        return self.head(q_out)


def find_stanford_checkpoint() -> Path:
    seen = set()
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if base in seen:
            continue
        seen.add(base)
        for p in (
            base / "stanford_cars" / "stanford_cars_resnet.pth",
            base / "stanford_cars_resnet.pth",
        ):
            if p.is_file():
                return p.resolve()
    raise FileNotFoundError(
        "stanford_cars_resnet.pth not found"
    )


FEATURE_DIM = 1024
ADAPTER_DIM = 128
n_qubits = 6
n_vqc_layers = 2

ckpt_path = find_stanford_checkpoint()
print("Loading:", ckpt_path)

try:
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
except TypeError:
    checkpoint = torch.load(ckpt_path, map_location=device)

if isinstance(checkpoint, dict) and "model_state" in checkpoint:
    state_dict = checkpoint["model_state"]
else:
    state_dict = checkpoint

model = StanfordResNet(in_channels=3, num_classes=num_classes).to(device)
model.load_state_dict(state_dict, strict=True)

baseline_head = nn.Linear(FEATURE_DIM, num_classes).to(device)
with torch.no_grad():
    baseline_head.weight.copy_(model.fc.weight)
    baseline_head.bias.copy_(model.fc.bias)

model.fc = HybridHead(
    FEATURE_DIM, num_classes, n_qubits, n_vqc_layers, adapter_dim=ADAPTER_DIM
).to(device)

for name, p in model.named_parameters():
    if not name.startswith("fc."):
        p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable parameters:", trainable)


Loading: C:\Users\godca\Downloads\Last sem\Hybrid-Quantum-Neural-Networks-for-Image-Classification\stanford_cars\stanford_cars_resnet.pth
trainable parameters: 133388


## Train HQNN head (feature cache + 20 epochs)


In [ ]:
USE_FEATURE_CACHE = True
PRECOMPUTE_BATCH = 64
HEAD_BATCH_SIZE = 64
CACHE_DIR = Path("VMMRdb") if Path("VMMRdb").is_dir() else Path(".")
train_cache = CACHE_DIR / "hqnn_stanford_cached_train.pt"
test_cache = CACHE_DIR / "hqnn_stanford_cached_test.pt"


def _torch_load_cpu(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def materialize_feature_caches():
    model.eval()
    pm = getattr(train_loader, "pin_memory", False)
    for base_loader, path in [
        (train_loader, train_cache),
        (test_loader, test_cache),
    ]:
        if path.exists():
            print("Using existing cache:", path)
            continue
        feats_chunks, y_chunks = [], []
        fast_loader = DataLoader(
            base_loader.dataset,
            batch_size=PRECOMPUTE_BATCH,
            shuffle=False,
            num_workers=0,
            pin_memory=pm,
        )
        print(
            f"Caching {path.name}: {len(fast_loader)} batches",
            flush=True,
        )
        with torch.no_grad():
            for bi, (x, y) in enumerate(
                tqdm(fast_loader, desc=f"ResNet features -> {path.name}")
            ):
                if bi == 0:
                    print("  first batch in, shape:", tuple(x.shape), flush=True)
                x = x.to(device, non_blocking=True)
                if device.type == "cuda":
                    with torch.autocast(device_type="cuda", dtype=torch.float16):
                        z = model.forward_features(x)
                else:
                    z = model.forward_features(x)
                feats_chunks.append(z.float().cpu())
                y_chunks.append(y)
        blob = {
            "feats": torch.cat(feats_chunks, dim=0),
            "labels": torch.cat(y_chunks, dim=0),
        }
        torch.save(blob, path)
        print("Wrote", path, blob["feats"].shape)


if USE_FEATURE_CACHE:
    materialize_feature_caches()
    tr = _torch_load_cpu(train_cache)
    te = _torch_load_cpu(test_cache)
    print("Train feats:", tr["feats"].shape, "Test feats:", te["feats"].shape, flush=True)
    train_loop_loader = DataLoader(
        TensorDataset(tr["feats"], tr["labels"]),
        batch_size=HEAD_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )
    test_loop_loader = DataLoader(
        TensorDataset(te["feats"], te["labels"]),
        batch_size=HEAD_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )
    active_net = model.fc
else:
    train_loop_loader, test_loop_loader = train_loader, test_loader
    active_net = model


def run_epoch(
    net,
    loader,
    opt: Optional[torch.optim.Optimizer],
    loss_fn,
    *,
    verbose: bool = False,
    show_pbar: bool = False,
):
    train = opt is not None
    net.train(train)
    total_loss = 0.0
    correct = 0
    n = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    if show_pbar and train:
        it = tqdm(loader, leave=True)
    else:
        it = loader
    if verbose:
        print(
            f"run_epoch ({'train' if train else 'eval'}): {len(loader)} batches",
            flush=True,
        )
    with ctx:
        for bi, (x, y) in enumerate(it):
            if verbose and bi == 0:
                print("  first batch:", tuple(x.shape), flush=True)
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            if train:
                opt.zero_grad()
            logits = net(x)
            loss = loss_fn(logits, y)
            if train:
                loss.backward()
                opt.step()
            total_loss += loss.item() * x.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            n += x.size(0)
            if train and show_pbar:
                it.set_postfix(loss=loss.item())
    return total_loss / n, correct / n


def should_log_epoch(epoch: int, total: int, first: int = 5, last: int = 5) -> bool:
    if total <= first + last:
        return True
    return epoch <= first or epoch > total - last


loss_fn = nn.CrossEntropyLoss()

if USE_FEATURE_CACHE:
    baseline_eval_net = baseline_head
else:

    class FrozenBackboneLinearBaseline(nn.Module):
        def __init__(self, backbone: nn.Module, head: nn.Module):
            super().__init__()
            self.backbone = backbone
            self.head = head

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            z = self.backbone.forward_features(x)
            return self.head(z)

    baseline_eval_net = FrozenBackboneLinearBaseline(model, baseline_head).to(device)

print(
    "=== Classical baseline (frozen backbone + pretrained Linear head, eval only) ===",
    flush=True,
)
baseline_eval_net.eval()
tr_loss, tr_acc = run_epoch(
    baseline_eval_net,
    train_loop_loader,
    None,
    loss_fn,
    verbose=False,
    show_pbar=False,
)
te_loss, te_acc = run_epoch(
    baseline_eval_net,
    test_loop_loader,
    None,
    loss_fn,
    verbose=False,
    show_pbar=False,
)
print(
    f"  train loss {tr_loss:.4f} acc {tr_acc:.4f}  test loss {te_loss:.4f} acc {te_acc:.4f}",
    flush=True,
)

print("=== HQNN (train hybrid head only) ===", flush=True)

epochs = 100
lr = 1e-3
LOG_FIRST, LOG_LAST = 5, 5
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, active_net.parameters()), lr=lr
)

hqnn_train_acc, hqnn_test_acc = [], []
middle_skipped = False
for epoch in range(1, epochs + 1):
    use_pbar = should_log_epoch(epoch, epochs, LOG_FIRST, LOG_LAST)
    tr_loss, tr_acc = run_epoch(
        active_net,
        train_loop_loader,
        optimizer,
        loss_fn,
        verbose=use_pbar,
        show_pbar=use_pbar,
    )
    te_loss, te_acc = run_epoch(
        active_net,
        test_loop_loader,
        None,
        loss_fn,
        verbose=use_pbar,
        show_pbar=False,
    )
    hqnn_train_acc.append(tr_acc)
    hqnn_test_acc.append(te_acc)
    if should_log_epoch(epoch, epochs, LOG_FIRST, LOG_LAST):
        print(
            f"epoch {epoch:02d}/{epochs}  train loss {tr_loss:.4f} acc {tr_acc:.4f}  "
            f"test loss {te_loss:.4f} acc {te_acc:.4f}",
            flush=True,
        )
    elif not middle_skipped:
        print(
            f"  ... omitting epoch logs {LOG_FIRST + 1}–{epochs - LOG_LAST} ...",
            flush=True,
        )
        middle_skipped = True

import matplotlib.pyplot as plt

epoch_axis = list(range(1, len(hqnn_train_acc) + 1))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epoch_axis, hqnn_train_acc, label="train accuracy")
ax.plot(epoch_axis, hqnn_test_acc, label="test accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("HQNN: accuracy vs epoch")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



Using existing cache: hqnn_stanford_cached_train.pt
Using existing cache: hqnn_stanford_cached_test.pt
Loading feature tensors into RAM...
Train feats: torch.Size([8144, 1024]) Test feats: torch.Size([8041, 1024])
=== Classical baseline (frozen backbone + pretrained Linear head, eval only) ===
  train loss 1.6735 acc 0.9794  test loss 2.4491 acc 0.7321
=== HQNN (train hybrid head only) ===


  0%|          | 0/128 [00:00<?, ?it/s]

run_epoch (train): 128 batches
  first batch: (64, 1024)


100%|██████████| 128/128 [00:06<00:00, 20.07it/s, loss=5.07]

run_epoch (eval): 126 batches
  first batch: (64, 1024)


...

  0%|          | 0/128 [00:00<?, ?it/s]

run_epoch (train): 128 batches
  first batch: (64, 1024)


100%|██████████| 128/128 [00:05<00:00, 22.29it/s, loss=2.48]

run_epoch (eval): 126 batches
  first batch: (64, 1024)


epoch 100/100  train loss 2.7856 acc 0.2143  test loss 3.6876 acc 0.1229


## save checkpoint


In [ ]:
# out = Path("stanford_hqnn_head.pth")
# torch.save({"model_state": model.state_dict(), "num_classes": num_classes, "n_qubits": n_qubits}, out)
# print("Saved", out)


NameError: name 'Path' is not defined